Agent 1:
Agents architectures:
  1. info extractor to json. pdf, txt, and doc
  2. job link info extractor to json
Task architectures:
  1. 
  2.


Agent 2:
Agents architectures:
  1. modify resume json output to json
  2. review agent
  3. critic agent
  4. cover_letter_strategy
  5. Interview preparation material: A document containing key questions and talking points that the candidate should prepare for the interview.

Task architectures:
  1.
  2.
  3.
  4.
  5.

Agent 3:
1. Resume Strategist
2. Document Validation Manager
3. xx

utils: 
pdf converter: convert json to pdf.
RAG optimizer: optimizing RAG architecture.



version 2:
optimize agents.
 

In [1]:
import logging
from pathlib import Path
from typing import Dict, Any
import docx2txt
import PyPDF2
from pdfminer.high_level import extract_text

# from pdfminer.high_level import extract_text
# from ..exceptions.custom_exceptions import ResumeParsingError
# from ..config.settings import ALLOWED_RESUME_FORMATS, MAX_FILE_SIZE
ALLOWED_RESUME_FORMATS = ['.pdf', '.docx', '.txt']
MAX_FILE_SIZE = 5 * 1024 * 1024  # 5MB

# Output directory
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

class JobFusionError(Exception):
    """Base exception class for JobFusion"""
    pass

class ResumeParsingError(JobFusionError):
    """Raised when there's an error parsing the resume"""
    pass

logger = logging.getLogger(__name__)

class ResumeParser:
    """Handle different resume formats and convert them to text"""
    
    @classmethod
    def parse_resume(cls, file_path: str) -> str:
        """Parse resume file in different formats"""
        try:
            file_path = Path(file_path)
            
            # Validate file
            cls._validate_file(file_path)
            
            # Parse based on extension
            if file_path.suffix == '.pdf':
                return cls._parse_pdf(file_path)
            elif file_path.suffix == '.docx':
                return cls._parse_docx(file_path)
            elif file_path.suffix == '.txt':
                return cls._parse_txt(file_path)
                
        except Exception as e:
            logger.error(f"Error parsing resume: {str(e)}")
            raise ResumeParsingError(f"Failed to parse resume: {str(e)}")

    @staticmethod
    def _validate_file(file_path: Path) -> None:
        """Validate file format and size"""
        if file_path.suffix.lower() not in ALLOWED_RESUME_FORMATS:
            raise ResumeParsingError(f"Unsupported file format: {file_path.suffix}")
            
        if file_path.stat().st_size > MAX_FILE_SIZE:
            raise ResumeParsingError("File size exceeds maximum limit")

    @staticmethod
    def _parse_pdf(file_path: Path) -> str:
        """Parse PDF files"""
        try:
            # Try pdfminer first
            text = extract_text(str(file_path))
            if text.strip():
                return text
                
            # Fallback to PyPDF2
            with open(file_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                return " ".join(page.extract_text() for page in pdf_reader.pages)
                
        except Exception as e:
            raise ResumeParsingError(f"Failed to parse PDF: {str(e)}")

    @staticmethod
    def _parse_docx(file_path: Path) -> str:
        """Parse DOCX files"""
        try:
            return docx2txt.process(str(file_path))
        except Exception as e:
            raise ResumeParsingError(f"Failed to parse DOCX: {str(e)}")

    @staticmethod
    def _parse_txt(file_path: Path) -> str:
        """Parse TXT files"""
        try:
            return file_path.read_text(encoding='utf-8')
        except Exception as e:
            raise ResumeParsingError(f"Failed to parse TXT: {str(e)}")

# utils/scrapers.py
import logging
from typing import Dict, Any
from urllib.parse import urlparse
from crewai_tools import ScrapeWebsiteTool
from ..exceptions.custom_exceptions import JobDescriptionError

logger = logging.getLogger(__name__)

class JobDescriptionScraper:
    """Scrape and analyze job descriptions"""
    
    def __init__(self):
        self.scraper = ScrapeWebsiteTool()

    def scrape_job_description(self, url: str) -> Dict[str, Any]:
        """Scrape and parse job description"""
        try:
            # Validate URL
            self._validate_url(url)
            
            # Scrape content
            content = self.scraper.scrape(url)
            
            # Parse content
            return self._parse_content(content)
            
        except Exception as e:
            logger.error(f"Error scraping job description: {str(e)}")
            raise JobDescriptionError(f"Failed to scrape job description: {str(e)}")

    @staticmethod
    def _validate_url(url: str) -> None:
        """Validate URL format"""
        try:
            result = urlparse(url)
            if not all([result.scheme, result.netloc]):
                raise JobDescriptionError("Invalid URL format")
        except Exception as e:
            raise JobDescriptionError(f"Invalid URL: {str(e)}")

    def _parse_content(self, content: str) -> Dict[str, Any]:
        """Parse job description content"""
        try:
            return {
                'requirements': self._extract_requirements(content),
                'skills': self._extract_skills(content),
                'experience': self._extract_experience(content),
                'education': self._extract_education(content)
            }
        except Exception as e:
            raise JobDescriptionError(f"Failed to parse job description: {str(e)}")

    # Add implementation for extraction methods
    def _extract_requirements(self, content: str) -> list:
        # Implementation for extracting requirements
        pass

    def _extract_skills(self, content: str) -> list:
        # Implementation for extracting skills
        pass

    def _extract_experience(self, content: str) -> list:
        # Implementation for extracting experience
        pass

    def _extract_education(self, content: str) -> list:
        # Implementation for extracting education
        pass





ModuleNotFoundError: No module named 'pdfminer.converter'

In [ ]:
import pdfminer
# from pdfminer.converter import TextConverter
# from pdfminer.high_level import extract_text

from pdfminer.converter import TextConverter

ModuleNotFoundError: No module named 'pdfminer.converter'

In [11]:
!pip uninstall pdfminer --yes
!pip install pdfminer.six


[notice] A new release of pip is available: 24.1.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [18]:
from pdfminer.high_level import extract_text


ModuleNotFoundError: No module named 'pdfminer.converter'

In [1]:
from dotenv import load_dotenv
from crewai import Crew, Task, Agent, Process
from crewai_tools import ScrapeWebsiteTool
from langchain.chat_models import ChatOpenAI
import os

load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')
llm_35_turbo = ChatOpenAI(api_key=openai_api_key, model='gpt-3.5-turbo', temperature=0.7)
manager_llm_35_turbo = ChatOpenAI(api_key=openai_api_key, model='gpt-3.5-turbo')

/Users/frankwei/Documents/Side_Project/jobfusion_subj/jobfusion310/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [16]:
import streamlit as st
from crewai import Crew
import logging
from crewai_tools import ScrapeWebsiteTool, DOCXSearchTool, SeleniumScrapingTool, MDXSearchTool, JSONSearchTool
from langchain.chat_models import ChatOpenAI
from textwrap import dedent
from config import configure as cfg
from datetime import datetime
# from agents import ResumeAgents
# from tasks import ResumeTasks

logger = logging.getLogger(__name__)
file_handler = logging.FileHandler(f'log/resume_enhancement_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
logger.addHandler(file_handler)

class ResumeAgents:
    def __init__(self, openai_api_key, jd_url, resume_input, valid_resume):
        self.llm = ChatOpenAI(api_key=openai_api_key, model='gpt-3.5-turbo', temperature=0.7)
        # self.llm = ChatOpenAI(api_key=openai_api_key, model='gpt-4o-mini', temperature=0.7)
        self.jd_url = jd_url
        self.resume_input = resume_input
        self.valid_resume = valid_resume
        logger.info("Initializing ResumeAgents")

    def create_info_extractor(self) -> Agent:
        try:
            return Agent(
                role=cfg.resume_info_extract_role,
                goal=cfg.resume_info_extract_agent_goal,
                backstory=cfg.resume_info_extract_agent_bkground,
                llm=self.llm,
                verbose=True,
                tools=[DOCXSearchTool(docx=self.resume_input)],
            )
        except Exception as e:
            logger.error(f"Error creating info extractor agent: {str(e)}")
            raise

    def resume_info_reviewer(self) -> Agent:
        try:
            return Agent(
                role='Information Extract Reviewer',
                goal='Review and validate extracted information from resume to make sure everything is correct, and make the correction if needed',
                backstory='Expert in reviewing and validating information extracted from resumes',
                llm=self.llm,
                verbose=True,
                tools=[DOCXSearchTool(docx=self.resume_input), JSONSearchTool(json_path=self.valid_resume)],
            )
        except Exception as e:
            logger.error(f"Error creating info extractor agent: {str(e)}")
            raise   

    def create_job_analyzer(self) -> Agent:
        try:
            return Agent(
                role='Job Description Analyzer',
                goal='Extract key requirements and qualifications from job postings',
                backstory='Specialist in analyzing job descriptions and identifying key requirements',
                llm=self.llm,
                verbose=True,
                tools=[ScrapeWebsiteTool(website_url=self.jd_url)]
            )
        except Exception as e:
            logger.error(f"Error creating job analyzer agent: {str(e)}")
            raise
    # def create_resume_optimizer(self) -> Agent:
    #     try:
    #         return Agent(
    #             role='Resume Optimizer',
    #             goal='Enhance resume content based on job requirements',
    #             backstory='Expert at aligning resumes with job requirements',
    #             llm=self.llm,
    #             verbose=True,
    #             # tools=['content_optimizer', 'keyword_matcher']
    #         )
    #     except Exception as e:
    #         logger.error(f"Error creating resume optimizer agent: {str(e)}")
    #         raise

class ResumeTasks:
    @staticmethod
    def create_extraction_task(agent, resume_file: str) -> Task:
        try:
            return Task(
                description=cfg.resume_Info_extract_task,
                agent=agent,
                expected_output=cfg.resume_info_task_expected_output,
                output_format="JSON",
                output_file='output/resume.json',
            )
        except Exception as e:
            logger.error(f"Error creating extraction task: {str(e)}")
            raise

    @staticmethod
    def extraction_valid_task(agent, resume_file: str, valid_resume:str) -> Task:
        try:
            return Task(
                description=f'valid extracted information from {valid_resume} and compare to {resume_file}',
                agent=agent,
                expected_output='A new JSON file',
                output_format="JSON",
                output_file='output/valid_resume.json',
            )
        except Exception as e:
            logger.error(f"Error creating extraction task: {str(e)}")
            raise

    @staticmethod
    def create_job_analysis_task(agent, job_url: str) -> Task:
        try:
            return Task(
                description=f"Analyze job posting from URL: {job_url}",
                agent=agent,
                expected_output="JSON formatted job requirements",
                output_format="JSON",
                output_file='output/job.json',
            )
        except Exception as e:
            logger.error(f"Error creating job analysis task: {str(e)}")
            raise

    

In [22]:
resume_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/inputs/resume_AI.docx'
job_url = 'https://www.amazon.jobs/en/jobs/2846094/principal-applied-scientist-amazon-prime'
valid_resume = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/reume.json'

resume_agents = ResumeAgents(openai_api_key, job_url, resume_file,valid_resume)
                # Create crew
crew = Crew(
    agents=[
        resume_agents.create_info_extractor(),
        # resume_agents.resume_info_reviewer(),
        resume_agents.create_job_analyzer(),
        # resume_agents.create_resume_optimizer()
    ],
    tasks=[
        ResumeTasks.create_extraction_task(resume_agents.create_info_extractor(), resume_file),
        # ResumeTasks.extraction_valid_task(resume_agents.resume_info_reviewer(), resume_file, valid_resume),
        ResumeTasks.create_job_analysis_task(resume_agents.create_job_analyzer(), job_url)
    ],
    verbose=True
)

# Execute tasks
result = crew.kickoff()

Inserting batches in chromadb:   0%|          | 0/1 [00:00<?, ?it/s]2024-12-16 15:37:09,524 - 8304408640 - sqlite.py-sqlite:284 - WARNING: Insert of existing embedding ID: default-app-id--f84ac5ef2f0d12c93d726dff546949c0760e177683e2c042a1340960eeb67a7c
2024-12-16 15:37:09,526 - 8304408640 - sqlite.py-sqlite:284 - WARNING: Insert of existing embedding ID: default-app-id--6fd78a686612a57f0518bd3b9becb7895ee735e48e470aa97b261b1bfb7d3cc4
2024-12-16 15:37:09,528 - 8304408640 - sqlite.py-sqlite:284 - WARNING: Insert of existing embedding ID: default-app-id--6c8defe9a06e20a7ff1a8f385a75687adcf960cbccca4aab6d2432d67603bd14
2024-12-16 15:37:09,529 - 8304408640 - sqlite.py-sqlite:284 - WARNING: Insert of existing embedding ID: default-app-id--d83bf957dd7e49b9ced3e2f8ae93412fda6af992a6ddc3814b4348b14b0dde26
2024-12-16 15:37:09,531 - 8304408640 - sqlite.py-sqlite:284 - WARNING: Insert of existing embedding ID: default-app-id--bc4a8ed0415f09c8037e86cd82e868c3dd9be9368393676a9585f3ac57e86491
2024-12

 [DEBUG]: == Working Agent: Resume information extracter
 [INFO]: == Starting Task: Extract information from the provided resume files including summary, work experience, education, skills, address,                      email address, and phone number to a json file. Make sure you are                          including all the working experiences. Name is always appearing in the first line of the resume without any prefix or name tag.                            When you are done with the work, take a deep breath and double check what you have done to make sure the output json file aligns with the schema including all the information.                            The schema you should be follow is: RESUME_SCHEMA = {
    "personal_info": {
        "name": str,
        "title": str,
        "summary": str,
        "contact": {
            "email": str,
            "phone": str,
            "location": str,
            "linkedin": str
        }
    },
    "work_experience": [{
        "compa

2024-12-16 15:37:19,317 - 8304408640 - manager.py-manager:282 - WARNING: Error in TokenCalcHandler.on_llm_start callback: KeyError('Could not automatically map gpt-4o-mini to a tokeniser. Please use `tiktoken.get_encoding` to explicitly get the tokeniser you expect.')


 

Relevant Content:
audiences", "Experience working with real-world data sets and building scalable models from big data", "Thinks strategically but stays on top of tactical execution", "Exhibits excellent business judgment; balances business, product, and technology very well", "Independent thinker who can make convincing, information-based arguments", "Recruit and groom high caliber talent"

"personal_info": { "name": "John Doe", "title": "Lead Quantitative Modeler", "summary": "Experienced Lead Quantitative Modeler with a background in applying NLP techniques, text summarization, and scoring algorithms. Skilled in Python, SQL, LangChain, crewAI, AWS, PyTorch, and more.", "contact": { "email": "johndoe@email.com", "phone": "123-456-7890", "location": "Washington D.C.", "linkedin": "linkedin.com/in/johndoe" "work_experience": [ "company": "Fannie Mae", "position": "Lead Quantitative Modeler", "duration": "June 2020 - August 2022", "location": "Washington D.C.", "achievements": [ "App

2024-12-16 15:37:23,076 - 8304408640 - manager.py-manager:282 - WARNING: Error in TokenCalcHandler.on_llm_start callback: KeyError('Could not automatically map gpt-4o-mini to a tokeniser. Please use `tiktoken.get_encoding` to explicitly get the tokeniser you expect.')


...

Thought: 
I have gathered the job description content from the provided URL. Now, I need to extract the key requirements and qualifications in a JSON format.

Action: Delegate work to co-worker
Action Input: {
    "coworker": "Job Description Analyzer",
    "task": "Extract key requirements and qualifications from job posting",
    "context": "The job posting is for the position of Principal Applied Scientist at Amazon Prime. Key responsibilities include owning and driving complex solutions, researching and deploying machine learning solutions, and mentoring senior scientists. Basic qualifications required include a Ph.D. in Machine Learning or related fields, 10+ years of post-Ph.D. experience, and expertise in machine learning and statistics. Preferred qualifications include extensive machine learning research experience and strong analytic skills."
}

> Entering new CrewAgentExecutor chain...
I need to gather detailed information about the job posting for the Principal Applied Scien

2024-12-16 15:38:10,436 - 8304408640 - manager.py-manager:282 - WARNING: Error in TokenCalcHandler.on_llm_start callback: KeyError('Could not automatically map gpt-4o-mini to a tokeniser. Please use `tiktoken.get_encoding` to explicitly get the tokeniser you expect.')


 

Principal Applied Scientist, Amazon Prime - Job ID: 2846094 | Amazon.jobs
Skip to main contentHomeTeamsLocationsJob categoriesMy careerMy applicationsMy profileAccount securitySettingsSign outResourcesDisability accommodationsBenefitsDiversity, equity, and inclusionInterview tipsLeadership principlesWorking at AmazonFAQ×Principal Applied Scientist, Amazon PrimeJob ID: 2846094 | Amazon.com Services LLCApply nowDESCRIPTIONAmazon Prime’s Science organization leads the Research & Development towards innovation for Amazon Prime. Amazon Prime is the backbone of Amazon’s consumer business and aspires to be the world’s most engaging, satisfying, and loved membership program, driving growth and profitability. The program serves over 200 million members across 25 countries and is key to Amazon’s customer growth and engagement. Prime Science innovates in Artificial Intelligence and Economics, to develop algorithms and systems for automated marketing, personalization, targeting, and decisioning

2024-12-16 15:38:16,576 - 8304408640 - manager.py-manager:282 - WARNING: Error in TokenCalcHandler.on_llm_start callback: KeyError('Could not automatically map gpt-4o-mini to a tokeniser. Please use `tiktoken.get_encoding` to explicitly get the tokeniser you expect.')


Thought: I now know the final answer  
Final Answer: 
**Key Requirements and Qualifications for Principal Applied Scientist at Amazon Prime:**

**Basic Qualifications:**
- Ph.D. in Machine Learning, Computer Science, or related disciplines.
- 10+ years of post-Ph.D. experience, delivering highly effective machine learning solutions that solve real-world problems at a very large scale.
- Expert in machine learning, statistics, and experiment design.
- Expert in taking machine learning solutions into production at a large scale.
- Experience with Large Language Models and Generative AI.
- Experience hiring and building teams of applied scientists.
- Proven communication and collaboration skills that enable you to earn trust at all levels of a large organization; ability to influence.

**Preferred Qualifications:**
- 10+ years of innovative machine learning research experience, with strong analytic and problem-solving skills.
- 10+ years of experience building and deploying groundbreaking

In [36]:
!ls /Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/inputs

contents            resume_AI.docx      skills_profile.docx


In [19]:
# class ResumeAgents:
#     def __init__(self, openai_api_key, jd_url, resume_input):
#         self.llm = ChatOpenAI(api_key=openai_api_key, model='gpt-3.5-turbo', temperature=0.7)
#         self.jd_url = jd_url
#         self.resume_input = resume_input
#         logger.info("Initializing ResumeAgents")
#     def create_resume_optimizer(self) -> Agent:
#         try:
#             return Agent(
#                 role='Resume Optimizer',
#                 goal='Enhance resume content based on job requirements',
#                 backstory='Expert at aligning resumes with job requirements',
#                 llm=self.llm,
#                 verbose=True,
#                 # tools=['content_optimizer', 'keyword_matcher'] 
#             )
#         except Exception as e:
#             logger.error(f"Error creating resume optimizer agent: {str(e)}")
#             raise

#     def cover_letter_strategist(self) -> Agent:
#         try:
#             return Agent(
#                     role='Cover Letter Strategist for Data Scientist',
#                     goal='Find all the best ways to write a cover letter stand out in the job market.',
#                     backstory=('''With a strategic mind and an eye for detail, you excel at highlight the \
#                         most relevant skills and experiences, ensuring they resonate perfectly with the job's requirements.'''),
#                     llm=llm_35_turbo,
#                     tools=[DOCXSearchTool(docx=self.resume_input)],
#                     allow_delegation=False,
#                     verbose=True,
#                     max_iter=5)
#         except Exception as e:
#             logger.error(f"Error creating cover letter agent: {str(e)}")
#             raise
    
#     def interview_preparer(self):
#         try:
#             return Agent(
#                 role='Data Scientist Interview Preparer',
#                 goal='Create interview quetions and talking points based on the resume and job requirements',
#                 backstory=("""Your role is crucial in anticipanting the dynamics of interviews. With your ability to formulate \
#                     key questions and talking points, you prepare candidates for success, ensuring they can confidently \
#                     address all aspects of the job they are applying for."""),
#                 llm=llm_35_turbo,
#                 allow_delegation=False,
#                 verbose=True,
#                 max_iter=5)
#         except Exception as e:
#             logger.error(f"Error creating interview preparer agent: {str(e)}")
#             raise

# class ResumeTasks:
#     @staticmethod
#     def job_optimizer_task(agent, job_url: str) -> Task:
#         try:
#             return Task(
#                 description=cfg.,
#                 agent=agent,
#                 expected_output="JSON formatted job requirements",
#                 output_format="json",
#                 output_file='output/updated_resume.json',
#             )
#         except Exception as e:
#             logger.error(f"Error creating job analysis task: {str(e)}")
#             raise

#     @staticmethod
#     def cover_letter_strategy_task(self, agent):
#         try:
#             return Task(description=dedent(f'''
#                 Using the profile, job requirements obtained from previous tasks, and the udpated resume to write a cover letter to apply the position within 4 paragraphs. 
#                 Make sure to emphasize the candidate's strengths but don't make up any information, and reflect the candidates abilities and how it matches the job posting.
#                 {self.__tip_section()}
#                 '''),
#                 expected_output=("A cover letter that effectively highlights the candidate's qualifications and experiences relevant to the job."),
#                 output_file='output/coverletter.md',
#                 agent=agent) 
#         except Exception as e:
#             logger.error(f"Error creating cover_letter task: {str(e)}")
#             raise
        
#     @staticmethod
#     def interview_preparation_task(self, agent):
#         try:
#             return Task(description=dedent(f'''
#                 Create a set of potential interview questions and talking points based on the latest updated resume and job requirements. 
#                 Utilize tools to generate relevant questions and discussion points. Make sure to use these question and talking points to help 
#                 the candidate highlight the main points of the resume and how it matches the job posting. 
#                 {self.__tip_section()}
#                 '''),
#                 expected_output=("A document containing key questions and talking points that the candidate should prepare for the interview."),
#                 output_file='output/interview_preparation_materials.txt',
#                 agent=agent) 
#         except Exception as e:
#             logger.error(f"Error creating job analysis task: {str(e)}")
#             raise

#     def __tip_section(self):
#         return "If you do your BEST WORK, I'll tip you $100!"

In [26]:
# enhancement_agents.py
import logging
from typing import Dict, Any
from crewai import Agent
from langchain.llms import OpenAI
from datetime import datetime
from crewai_tools import ScrapeWebsiteTool, DOCXSearchTool, SeleniumScrapingTool, MDXSearchTool, JSONSearchTool
logger = logging.getLogger(__name__)
file_handler = logging.FileHandler(f'log/resume_enhancement_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
logger.addHandler(file_handler)

class EnhancementAgents:
    def __init__(self, openai_api_key, resume_input, jd_input):
        self.llm = ChatOpenAI(api_key=openai_api_key, model='gpt-3.5-turbo', temperature=0.7)
        self.jd_input = jd_input
        self.resume_input = resume_input
        logger.info("Initializing Enhancement Agents")

    def create_resume_modifier(self) -> Agent:
        try:
            return Agent(
                role='Resume Modifier',
                goal=f'''Enhance resume content with missing keywords and skills from job description. Keep the schema of the JSON file:
    {cfg.RESUME_SCHEMA}
                  only change the last layer for each section.''',
                backstory='''Expert at analyzing job requirements and modifying resumes to highlight 
                            relevant experience while maintaining authenticity''',
                llm=self.llm,
                tools=[JSONSearchTool(json_path=self.resume_input), JSONSearchTool(json_path=self.jd_input)],
                verbose=True
            )
        except Exception as e:
            logger.error(f"Error creating resume modifier agent: {str(e)}")
            raise

    def create_review_agent(self) -> Agent:
        try:
            return Agent(
                role='Resume Reviewer',
                goal=f'''Review and validate resume modifications for accuracy and relevance from create_resume_modifier. Keep the schema of the JSON file:\
    {cfg.RESUME_SCHEMA}
                  only change the last layer for each section.''',
                backstory='''Experienced resume reviewer with expertise in ensuring 
                            modifications maintain authenticity while maximizing impact''',
                llm=self.llm,
                # tools=['content_validator', 'consistency_checker'],
                verbose=True
            )
        except Exception as e:
            logger.error(f"Error creating review agent: {str(e)}")
            raise

    def create_critic_agent(self) -> Agent:
        try:
            return Agent(
                role='Resume Critic',
                goal=f'''Critically analyze resume for improvements and optimization opportunities. Keep the schema of the JSON file:\
    {cfg.RESUME_SCHEMA}
                  only change the last layer for each section.''',
                backstory='''Critical thinking expert specializing in identifying areas 
                            for resume enhancement and optimization''',
                llm=self.llm,
                # tools=['content_analyzer', 'improvement_suggester'],
                verbose=True
            )
        except Exception as e:
            logger.error(f"Error creating critic agent: {str(e)}")
            raise

    def create_cover_letter_strategist(self) -> Agent:
        try:
            return Agent(
                role='Cover Letter Strategist',
                goal='Create compelling, tailored cover letters that highlight relevant experience',
                backstory='''Professional writer specialized in crafting engaging cover letters 
                            that effectively communicate candidate qualifications''',
                llm=self.llm,
                # tools=['content_generator', 'tone_analyzer'],
                verbose=True
            )
        except Exception as e:
            logger.error(f"Error creating cover letter strategist: {str(e)}")
            raise

    def create_interview_prep_agent(self) -> Agent:
        try:
            return Agent(
                role='Interview Preparation Specialist',
                goal='Prepare comprehensive interview materials and talking points',
                backstory='''Interview preparation expert skilled at creating targeted questions 
                            and effective talking points''',
                llm=self.llm,
                # tools=['question_generator', 'talking_points_creator'],
                verbose=True
            )
        except Exception as e:
            logger.error(f"Error creating interview prep agent: {str(e)}")
            raise

# enhancement_tasks.py
from crewai import Task
from typing import Dict
import json

class EnhancementTasks:
    @staticmethod
    def create_modification_task(agent, resume_json: Dict, job_json: Dict) -> Task:
        try:
            logger.info("Creating resume modification task")
            return Task(
                description='''Analyze job requirements and modify resume to include relevant 
                              keywords, responsibilities, and skills while maintaining authenticity''',
                agent=agent,
                # context={
                #     "resume_data": resume_json,
                #     "job_data": job_json
                # },
                expected_output="Enhanced JSON resume with incorporated keywords and skills",
                output_format="json",
                output_file='output/enhanced_resume.json'
            )
        except Exception as e:
            logger.error(f"Error creating modification task: {str(e)}")
            raise

    @staticmethod
    def create_review_task(agent, modified_resume: Dict) -> Task:
        try:
            logger.info("Creating resume review task")
            return Task(
                description='''Review modified resume for accuracy, relevance, and authenticity. 
                              Suggest and implement necessary improvements''',
                agent=agent,
                # context={"modified_resume": modified_resume},
                expected_output="Reviewed and validated JSON resume",
                output_format="json",
                output_file='output/reviewed_resume.json'
            )
        except Exception as e:
            logger.error(f"Error creating review task: {str(e)}")
            raise

    @staticmethod
    def create_critique_task(agent, reviewed_resume: Dict) -> Task:
        try:
            logger.info("Creating resume critique task")
            return Task(
                description='''Critically analyze the resume for potential improvements 
                              and optimization opportunities''',
                agent=agent,
                # context={"reviewed_resume": reviewed_resume},
                expected_output="Critiqued and optimized JSON resume with improvement suggestions",
                output_format="json",
                output_file='output/final_update_resume.json'
            )
        except Exception as e:
            logger.error(f"Error creating critique task: {str(e)}")
            raise

    @staticmethod
    def create_cover_letter_task(agent, final_resume: Dict, job_json: Dict) -> Task:
        try:
            logger.info("Creating cover letter task")
            return Task(
                description='''Create a compelling 4-paragraph cover letter that highlights relevant 
                              experience and matches job requirements. Maintain authenticity while 
                              emphasizing candidate strengths''',
                agent=agent,
                # context={
                #     "final_resume": final_resume,
                #     "job_requirements": job_json
                # },
                expected_output="Professional cover letter in markdown format",
                output_format="markdown",
                output_file='output/cover_letter.md'
            )
        except Exception as e:
            logger.error(f"Error creating cover letter task: {str(e)}")
            raise

    @staticmethod
    def create_interview_prep_task(agent, final_resume: Dict, job_json: Dict) -> Task:
        try:
            logger.info("Creating interview preparation task")
            return Task(
                description='''Generate relevant interview questions and talking points based on 
                              resume and job requirements. Focus on highlighting key experiences 
                              and qualifications''',
                agent=agent,
                # context={
                #     "final_resume": final_resume,
                #     "job_requirements": job_json
                # },
                expected_output="Interview preparation guide with questions and talking points",
                output_format="markdown",
                output_file='output/interview_prep.md'
            )
        except Exception as e:
            logger.error(f"Error creating interview prep task: {str(e)}")
            raise

def save_outputs(results: Dict[str, Any], filename: str = "enhancement_outputs.json"):
    try:
        logger.info(f"Saving outputs to {filename}")
        with open(filename, 'w') as f:
            json.dump(results, f, indent=2)
    except Exception as e:
        logger.error(f"Error saving outputs: {str(e)}")
        raise

In [28]:
resume_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/resume.json'
job_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/job.json'
modified_resume_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/enhanced_resume.json'
reviewed_resume_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/reviewed_resume.json'
final_resume_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/final_resume.json'


resume_agents = EnhancementAgents(openai_api_key, resume_file, job_file)
                # Create crew
crew = Crew(
    agents=[
        resume_agents.create_resume_modifier(),
        resume_agents.create_review_agent(),
        resume_agents.create_critic_agent(),
        resume_agents.create_cover_letter_strategist(),
        resume_agents.create_interview_prep_agent()
    ],
    tasks=[
        EnhancementTasks.create_modification_task(resume_agents.create_resume_modifier(), resume_file, job_file),
        EnhancementTasks.create_review_task(resume_agents.create_review_agent(), modified_resume_file),
        EnhancementTasks.create_critique_task(resume_agents.create_critic_agent(), reviewed_resume_file),
        EnhancementTasks.create_cover_letter_task(resume_agents.create_cover_letter_strategist(), final_resume_file, job_file),
        EnhancementTasks.create_interview_prep_task(resume_agents.create_interview_prep_agent(), final_resume_file, job_file)
    ],
    verbose=True
)

# Execute tasks
result = crew.kickoff()

Inserting batches in chromadb: 100%|██████████| 1/1 [00:04<00:00,  4.16s/it]
2024-12-16 15:39:27,135 - 8304408640 - __init__.py-__init__:531 - WARNING: Overriding of current TracerProvider is not allowed


 [DEBUG]: == Working Agent: Resume Modifier
 [INFO]: == Starting Task: Analyze job requirements and modify resume to include relevant 
                              keywords, responsibilities, and skills while maintaining authenticity


> Entering new CrewAgentExecutor chain...
I need to start by analyzing the job requirements to identify the relevant keywords, responsibilities, and skills that need to be incorporated into the resume.

Action: Search a JSON's content
Action Input: {"search_query": "job requirements"} 

Relevant Content:
non-technical and technical audiences.", "Experience working with real-world data sets and building scalable models from big data.", "Thinks strategically, but stays on top of tactical execution.", "Exhibits excellent business judgment; balances business, product, and technology very well.", "Independent thinker who can make convincing, information-based arguments with a strong bias for action.", "Ability to work equally well with science, engineering, 

In [31]:
import json
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, ListItem, ListFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
import os
import sys
import logging
import streamlit as st
import docx2txt
from crewai import Crew, Task, Agent, Process
from crewai_tools import ScrapeWebsiteTool
# from jobfusion_agents import JobFusion_Agents
# from jobfusion_tasks import JobFusion_Tasks
# from jobfusion2_agents import JobFusion2_Agents
# from jobfusion2_tasks import JobFusion2_Tasks
# from langchain.chat_models import ChatOpenAI
from dotenv import load_dotenv
from config import configure as cfg
# from mock_interview_chatbot import *
# import streamlit as st

import os
import requests
from crewai import Agent, Task, Crew, Process
from langchain.llms import OpenAI
import markdown
import PyPDF2
from fpdf import FPDF

/Users/frankwei/Documents/Side_Project/jobfusion_subj/jobfusion310/lib/python3.10/site-packages/fpdf/__init__.py:39: UserWarning: You have both PyFPDF & fpdf2 installed. Both packages cannot be installed at the same time as they share the same module namespace. To only keep fpdf2, run: pip uninstall --yes pypdf && pip install --upgrade fpdf2
  warnings.warn(


In [32]:


def create_pdf_from_json(json_data, output_filename):
    '''
    TO DO: 
    1. Adding different format corresponding to different input resumes
    2. 
    '''
    if isinstance(json_data, str):
        with open(json_data, 'r') as file:
            json_data = json.load(file)

    # if isinstance(json_data, str):
    #     json_data = json.loads(json_data)
    
    doc = SimpleDocTemplate(output_filename, pagesize=letter,
                          rightMargin=72, leftMargin=72,
                          topMargin=72, bottomMargin=18)
    
    styles = getSampleStyleSheet()
    story = []
    
    styles.add(ParagraphStyle(name='Name',
                            fontSize=24,
                            spaceAfter=12))  # Increased space after name
    styles.add(ParagraphStyle(name='Contact',
                            fontSize=12,
                            spaceAfter=20))
    styles.add(ParagraphStyle(name='Section',
                            fontSize=16,
                            spaceBefore=20,
                            spaceAfter=12))
    
    # Personal Info
    if isinstance(json_data.get("personal_info"), dict):
        person = json_data["personal_info"]
        story.append(Paragraph(person.get("name", "No Name Provided"), styles["Name"]))
        
        if isinstance(person.get("contact"), dict):
            contact = person["contact"]
            contact_items = []
            if contact.get('email'): contact_items.append(contact['email'])
            if contact.get('phone'): contact_items.append(contact['phone'])
            if contact.get('location'): contact_items.append(contact['location'])
            if contact.get('linkedin'): contact_items.append(contact['linkedin'])
            
            contact_text = " | ".join(contact_items) if contact_items else "No Contact Information Provided"
            story.append(Paragraph(contact_text, styles["Contact"]))
        
        # story.append(Paragraph(person.get("title", "No Title Provided"), styles["Heading2"]))
        story.append(Paragraph('Summary', styles["Heading2"]))
        story.append(Paragraph(person.get("summary", "No Summary Provided"), styles["Normal"]))
    
    story.append(Spacer(1, 20))
    
    # Work Experience
    story.append(Paragraph("Work Experience", styles["Section"]))
    if isinstance(json_data.get("work_experience"), list) and json_data["work_experience"]:
        for job in json_data["work_experience"]:
            job_title = []
            if job.get('position'): job_title.append(f"<b>{job['position']}</b>")
            if job.get('company'): job_title.append(job['company'])
            job_title_text = " - ".join(job_title) if job_title else "Position Details Not Available"
            story.append(Paragraph(job_title_text, styles["Normal"]))
            
            location_info = []
            if job.get('duration'): location_info.append(job['duration'])
            if job.get('location'): location_info.append(job['location'])
            location_text = " | ".join(location_info) if location_info else "Location/Duration Not Available"
            story.append(Paragraph(location_text, styles["Normal"]))
            
            if isinstance(job.get("achievements"), list) and job["achievements"]:
                achievements = [ListItem(Paragraph(item, styles["Normal"])) 
                              for item in job["achievements"] if item]
                if achievements:
                    story.append(ListFlowable(achievements, bulletType='bullet'))
            story.append(Spacer(1, 12))
    else:
        story.append(Paragraph("No work experience provided", styles["Normal"]))
    
    # Education with reordered fields
    story.append(Paragraph("Education", styles["Section"]))
    if isinstance(json_data.get("education"), list) and json_data["education"]:
        for edu in json_data["education"]:
            # First line: Degree
            if edu.get('degree'):
                story.append(Paragraph(f"<b>{edu['degree']}</b>", styles["Normal"]))
            
            # Second line: Duration, School, Location
            edu_details = []
            if edu.get('duration'): edu_details.append(edu['duration'])
            if edu.get('school'): edu_details.append(edu['school'])
            if edu.get('location'): edu_details.append(edu['location'])
            
            details_text = " | ".join(edu_details) if edu_details else "Education Details Not Available"
            story.append(Paragraph(details_text, styles["Normal"]))
            story.append(Spacer(1, 8))
    else:
        story.append(Paragraph("No education information provided", styles["Normal"]))
    
    # Skills
    story.append(Paragraph("Skills", styles["Section"]))
    if isinstance(json_data.get("skills"), list) and json_data["skills"]:
        skills = [skill for skill in json_data["skills"] if skill]
        skills_text = "; ".join(skills) if skills else "No specific skills listed"
        story.append(Paragraph(skills_text, styles["Normal"]))
    else:
        story.append(Paragraph("No skills provided", styles["Normal"]))
    
    doc.build(story)

In [33]:
json_data = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/final_update_resume.json'
output_filename = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/resume.pdf'
create_pdf_from_json(json_data, output_filename)

In [49]:
json_data = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/updated_resumev7.json'
# json_data = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/final_update_resume.json'

with open(json_data, 'r') as file:
    json_data = json.load(file)

In [50]:
json_data

{'personal_info': {'name': 'Diana Liu',
  'title': 'AI Data Scientist Freelancer',
  'summary': 'Proficient in mathematical statistics, econometrics, machine learning and deep learning. Offering 15 years of extensive project management expertise with a deep involvement in experimental design, data integration and cleansing, feature engineering, as well as mastery in machine learning, optimization, and deep learning implementations. With extensive experience in leveraging advanced AI technologies, I have successfully utilized prompt engineering, Retrieval-Augmented Generation (RAG), and agents to develop robust applications that address complex business challenges, automate processes, and drive business growth.',
  'contact': {'email': 'lxjiao0805@gmail.com',
   'phone': '202.739.1368',
   'location': 'Washington D.C.',
   'linkedin': ''}},
 'work_experience': [{'company': 'AI Data Scientist Freelancer',
   'position': 'Freelancer',
   'duration': 'March 2023 - present',
   'location': 

In [1]:
from agents import ResumeAgents, EnhancementAgents
from tasks import ResumeTasks, EnhancementTasks


from dotenv import load_dotenv
from crewai import Crew, Task, Agent, Process
from crewai_tools import ScrapeWebsiteTool
from langchain.chat_models import ChatOpenAI
import os

load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')
# llm_35_turbo = ChatOpenAI(api_key=openai_api_key, model='gpt-3.5-turbo', temperature=0.7)
# manager_llm_35_turbo = ChatOpenAI(api_key=openai_api_key, model='gpt-3.5-turbo')


/Users/frankwei/Documents/Side_Project/jobfusion_subj/jobfusion310/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [2]:
resume_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/inputs/resume_AI.docx'
job_url = 'https://www.amazon.jobs/en/jobs/2846094/principal-applied-scientist-amazon-prime'
valid_resume = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/reume.json'

resume_agents = ResumeAgents(openai_api_key, job_url, resume_file,valid_resume)
                # Create crew
crew = Crew(
    agents=[
        resume_agents.create_info_extractor(),
        # resume_agents.resume_info_reviewer(),
        resume_agents.create_job_analyzer(),
        # resume_agents.create_resume_optimizer()
    ],
    tasks=[
        ResumeTasks.create_extraction_task(resume_agents.create_info_extractor(), resume_file),
        # ResumeTasks.extraction_valid_task(resume_agents.resume_info_reviewer(), resume_file, valid_resume),
        ResumeTasks.create_job_analysis_task(resume_agents.create_job_analyzer(), job_url)
    ],
    verbose=True
)

# Execute tasks
result = crew.kickoff()

Inserting batches in chromadb:   0%|          | 0/1 [00:00<?, ?it/s]2025-01-02 05:48:01,338 - 8204646464 - sqlite.py-sqlite:284 - WARNING: Insert of existing embedding ID: default-app-id--f84ac5ef2f0d12c93d726dff546949c0760e177683e2c042a1340960eeb67a7c
2025-01-02 05:48:01,340 - 8204646464 - sqlite.py-sqlite:284 - WARNING: Insert of existing embedding ID: default-app-id--6fd78a686612a57f0518bd3b9becb7895ee735e48e470aa97b261b1bfb7d3cc4
2025-01-02 05:48:01,341 - 8204646464 - sqlite.py-sqlite:284 - WARNING: Insert of existing embedding ID: default-app-id--6c8defe9a06e20a7ff1a8f385a75687adcf960cbccca4aab6d2432d67603bd14
2025-01-02 05:48:01,341 - 8204646464 - sqlite.py-sqlite:284 - WARNING: Insert of existing embedding ID: default-app-id--d83bf957dd7e49b9ced3e2f8ae93412fda6af992a6ddc3814b4348b14b0dde26
2025-01-02 05:48:01,342 - 8204646464 - sqlite.py-sqlite:284 - WARNING: Insert of existing embedding ID: default-app-id--bc4a8ed0415f09c8037e86cd82e868c3dd9be9368393676a9585f3ac57e86491
2025-01

 [DEBUG]: == Working Agent: Resume information extracter
 [INFO]: == Starting Task: Extract information from the provided resume files including summary, work experience, education, skills, address,                      email address, and phone number to a json file. Make sure you are                          including all the working experiences. Name is always appearing in the first line of the resume without any prefix or name tag.                            When you are done with the work, take a deep breath and double check what you have done to make sure the output json file aligns with the schema including all the information.                            The schema you should be follow is: RESUME_SCHEMA = {
    "personal_info": {
        "name": str,
        "title": str,
        "summary": str,
        "contact": {
            "email": str,
            "phone": str,
            "location": str,
            "linkedin": str
        }
    },
    "work_experience": [{
        "compa

In [3]:
resume_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/resume.json'
job_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/job.json'
modified_resume_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/enhanced_resume.json'
reviewed_resume_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/reviewed_resume.json'
final_resume_file = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/jobfusion_production/output/final_resume.json'

resume_agents = EnhancementAgents(openai_api_key, resume_file, job_file)
                # Create crew
crew = Crew(
    agents=[
        resume_agents.create_resume_modifier(),
        resume_agents.create_review_agent(),
        resume_agents.create_critic_agent(),
        resume_agents.create_cover_letter_strategist(),
        resume_agents.create_interview_prep_agent()
    ],
    tasks=[
        EnhancementTasks.create_modification_task(resume_agents.create_resume_modifier(), resume_file, job_file),
        EnhancementTasks.create_review_task(resume_agents.create_review_agent(), modified_resume_file),
        EnhancementTasks.create_critique_task(resume_agents.create_critic_agent(), reviewed_resume_file),
        EnhancementTasks.create_cover_letter_task(resume_agents.create_cover_letter_strategist(), final_resume_file, job_file),
        EnhancementTasks.create_interview_prep_task(resume_agents.create_interview_prep_agent(), final_resume_file, job_file)
    ],
    verbose=True
)

# Execute tasks
result = crew.kickoff()

Inserting batches in chromadb: 100%|██████████| 1/1 [00:02<00:00,  2.04s/it]
2025-01-02 05:49:32,800 - 8204646464 - __init__.py-__init__:531 - WARNING: Overriding of current TracerProvider is not allowed


 [DEBUG]: == Working Agent: Resume Modifier
 [INFO]: == Starting Task: Analyze job requirements and modify resume to include relevant 
                              keywords, responsibilities, and skills while maintaining authenticity


> Entering new CrewAgentExecutor chain...
I need to analyze the job requirements and modify the resume to include relevant keywords, responsibilities, and skills while maintaining authenticity.

Action: Search a JSON's content
Action Input: {"search_query": "job requirements"} 

Relevant Content:
execute solutions that continually delight customers.", "Hire, mentor, and guide senior scientists; partner with engineering leaders to build efficient and scalable solutions." "basic_qualifications": { "phd_disciplines": [ "Machine Learning", "Computer Science", "related disciplines" "experience_years": 10, "skills": [ "machine learning", "statistics", "experiment design", "Large Language Models", "Generative AI", "team building" "communication_skills": "Prove